In [26]:
import base64
import json
import os
import random
import re
import csv
import time
import urllib.error
import urllib.parse
import urllib.request
import traceback
from pathlib import Path
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed


import pandas as pd
from tqdm.notebook import tqdm
from litellm import completion
import litellm

# Input from Generate Fakes.ipynb
RUN_NUMBER = 6632473800
OUTPUT_PREFIX = "even_even_easier_generated_fakes_jun15"
INPUT_CSV_FILE = f"{OUTPUT_PREFIX}_{RUN_NUMBER}.csv"

# Output artifacts
OUTPUT_JSON_FILE = f"{OUTPUT_PREFIX}_with_commons_matches_{RUN_NUMBER}.json"
OUTPUT_CSV_FILE = f"{OUTPUT_PREFIX}_with_commons_matches_{RUN_NUMBER}.csv"
OUTPUT_CSV_FILE_WITH_ATRIB = f"{OUTPUT_PREFIX}_with_commons_matches_and_atrib_{RUN_NUMBER}.csv"

DOWNLOAD_DIR = Path("downloaded_commons_images")

# LiteLLM + Ollama settings
OLLAMA_API_BASE = os.getenv("OLLAMA_API_BASE", "http://localhost:11434")
#TEXT_MODEL = os.getenv("COMMONS_TEXT_MODEL", "ollama/qwen3.5:9b")
TEXT_MODEL = os.getenv("COMMONS_TEXT_MODEL", "ollama/gemma3n")

VISION_MODEL = os.getenv("COMMONS_VISION_MODEL", "ollama/gemma4:e4b")
#VISION_MODEL = os.getenv("COMMONS_VISION_MODEL", "ollama/qwen3.5:9b")
#VISION_MODEL = os.getenv("COMMONS_VISION_MODEL", "ollama/gemma3n")

# Use one standard browser-like UA for all outbound HTTP requests
STANDARD_USER_AGENT = (
    "Moonflower "
    "Daniel Erenrich (derenrich@wikimedia.org)"
)

# Search/evaluation settings
MAX_ROWS = 1000
MAX_ROUNDS = 2
QUERIES_PER_ROUND = 4
CANDIDATES_PER_QUERY = 3
MAX_VISION_EVALS_PER_ROUND = CANDIDATES_PER_QUERY * QUERIES_PER_ROUND
ACCEPT_SCORE = 70
REQUEST_DELAY_SECONDS = 0.15

DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

def _extract_json(text: str) -> dict:
    text = (text or "").strip()
    text = text.replace("▁", " ")
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\\s*", "", text)
        text = re.sub(r"\\s*```$", "", text)
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not match:
        raise ValueError("No JSON object found in model response.", text)
    try:
        return json.loads(match.group(0))
    except:
        print("parse fail", text, "----", match.group(0))
        raise 


def llm_json(model: str, prompt: str) -> dict:
    response = completion(
        model=model,
        api_base=OLLAMA_API_BASE,
        messages=[    {"role": "system", "content": "You are a helpful assistant designed to output only JSON."},
{"role": "user", "content": prompt}],
        reasoning_effort=None#"low"
    )
    text = response["choices"][0]["message"]["content"]
    return _extract_json(text)


def plan_queries(image_prompt: str, title: str = "", topic: str = "", n: int = 6) -> list[str]:
    prompt = f"""You create Wikimedia Commons search queries for image discovery.
Task: Generate {n} concise search queries likely to find real photos/illustrations matching the concept.
Input concept: {image_prompt}
Optional context title: {title}
Optional context topic: {topic}

Rules:
- Prefer generic visual nouns, scene descriptors, object classes, style terms (photo, illustration, engraving, portrait).
- Consider naming a thing you know exists that looks like this but it must be obscure if so (e.g. name a real but obscure lighthouse)
- Keep each query 2-4 words.
- Include a mix of literal and broader fallback phrasing.
- Return JSON only in this format {{\"queries\": [\"...\"]}} (do not output markdown or anything decorative)

Answer below with nothing else
"""
    data = llm_json(TEXT_MODEL, prompt)
    queries = [q.strip() for q in data.get("queries", []) if isinstance(q, str) and q.strip()]
    deduped = []
    for q in queries:
        if q.lower() not in {x.lower() for x in deduped}:
            deduped.append(q)
    return deduped[:n]


def refine_queries(
    image_prompt: str,
    tried_queries: list[str],
    best_reason: str,
    title: str = "",
    topic: str = "",
    n: int = 4,
) -> list[str]:
    tried_blob = "\n".join(f"- {q}" for q in tried_queries[-20:])
    prompt = f"""You are refining Wikimedia Commons search queries after weak image matches.
Concept: {image_prompt}
Title context: {title}
Topic context: {topic}
Why previous results were weak: {best_reason}

Already tried queries:
{tried_blob}

Generate {n} NEW concise alternative queries that are more likely to match visually.
Use more generic terms and adjacent concepts when needed.
Return JSON only with this format {{\"queries\": [\"...\"]}}

Answer below with nothing else
"""
    data = llm_json(TEXT_MODEL, prompt)
    tried_lc = {q.lower() for q in tried_queries}
    queries = []
    for q in data.get("queries", []):
        if isinstance(q, str) and q.strip() and q.lower() not in tried_lc:
            queries.append(q.strip())
    return queries[:n]


def commons_search_files(query: str, limit: int = 16) -> list[dict]:
    params = {
        "action": "query",
        "format": "json",
        "generator": "search",
        "gsrsearch": query,
        "gsrnamespace": 6,  # File namespace
        "gsrlimit": str(limit),
        "prop": "imageinfo",
        "iiprop": "url|size|mime|extmetadata",
        "iiurlwidth": "640",
    }
    url = "https://commons.wikimedia.org/w/api.php?" + urllib.parse.urlencode(params)

    try:
        req = urllib.request.Request(
            url,
            headers={
                "User-Agent": STANDARD_USER_AGENT,
                "Accept": "application/json",
            },
        )
        with urllib.request.urlopen(req, timeout=25) as resp:
            data = json.loads(resp.read().decode("utf-8", errors="replace"))
    except urllib.error.URLError:
        return []

    pages = (data.get("query") or {}).get("pages") or {}
    out = []
    for p in pages.values():
        info = (p.get("imageinfo") or [{}])[0]        
        file_url = info.get("url")        
        if file_url and (file_url.endswith(".pdf") or file_url.endswith(".mp3") or file_url.endswith(".wav") or file_url.endswith(".mp3") or file_url.endswith(".tiff") or file_url.endswith(".djvu") or file_url.endswith(".ogg") or file_url.endswith(".ogv") or file_url.endswith("webm")):
            continue
        thumb_url = info.get("thumburl") or file_url
        if not file_url:
            continue
        out.append(
            {
                "commons_title": p.get("title", ""),
                "pageid": p.get("pageid"),
                "file_url": file_url,
                "thumb_url": thumb_url,
                "mime": info.get("mime", ""),
                "width": info.get("width"),
                "height": info.get("height"),
            }
        )
    return out


def _url_to_ext(url: str) -> str:
    ext = Path(urllib.parse.urlparse(url).path).suffix.lower()
    if ext in {".jpg", ".jpeg", ".png", ".webp", ".gif"}:
        return ext
    return ".jpg"


def download_image(url: str, out_dir: Path, file_stem: str) -> Path | None:
    ext = _url_to_ext(url)
    target = out_dir / f"{file_stem}{ext}"
    try:
        req = urllib.request.Request(
            url,
            headers={
                "User-Agent": STANDARD_USER_AGENT,
                "Accept": "image/avif,image/webp,image/*,*/*;q=0.8",
            },
        )
        with urllib.request.urlopen(req, timeout=30) as resp:
            data = resp.read()
        if not data:
            return None
        target.write_bytes(data)
        return target
    except Exception:
        return None


def image_file_to_data_url(path: Path) -> str:
    b = path.read_bytes()
    b64 = base64.b64encode(b).decode("ascii")
    mime = "image/jpeg"
    ext = path.suffix.lower()
    if ext == ".png":
        mime = "image/png"
    elif ext == ".webp":
        mime = "image/webp"
    elif ext == ".gif":
        mime = "image/gif"
    return f"data:{mime};base64,{b64}"


def vision_score_candidate(image_path: Path, image_prompt: str, title: str = "", topic: str = "") -> dict:
    rubric = f"""You are scoring whether an image matches a prompt for a fake-article card.
Prompt to match: {image_prompt}
Title context: {title}
Topic context: {topic}

Score guidelines:
- 90-100: very strong match (every described element is present)
- 70-89: usable match with minor mismatch
- 40-69: weak/partial match
- 0-39: poor or unrelated

Return JSON only with this schema:
{{
  \"detected_subject\": \"what the image appears to show\",
  \"reason\": \"short reason\",
  \"score\": 0
}}

Do not use markdown or any other formatting.

Answer below with nothing else
"""
    data_url = image_file_to_data_url(image_path)
    resp = completion(
        model=VISION_MODEL,
        api_base=OLLAMA_API_BASE,
        messages=[            
            {"role": "system", "content": "You are a helpful assistant designed to output only JSON."},
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": rubric},
                    {"type": "image_url", "image_url": {"url": data_url}},
                ],
            }
        ]
    )
    text = resp["choices"][0]["message"]["content"]
    parsed = _extract_json(text)

    score = parsed.get("score", 0)
    try:
        score = int(round(float(score)))
    except Exception:
        score = 0
    score = max(0, min(100, score))

    return {
        "score": score,
        "reason": str(parsed.get("reason", "")).strip(),
        "detected_subject": str(parsed.get("detected_subject", "")).strip(),
    }


def find_best_commons_image(
    image_prompt: str,
    title: str = "",
    topic: str = "",
    row_index: int = 0,
    max_rounds: int = MAX_ROUNDS,
    queries_per_round: int = QUERIES_PER_ROUND,
    candidates_per_query: int = CANDIDATES_PER_QUERY,
    max_vision_evals_per_round: int = MAX_VISION_EVALS_PER_ROUND,
    accept_score: int = ACCEPT_SCORE,
    delay_seconds: float = REQUEST_DELAY_SECONDS,
):
    all_tried_queries = []
    seen_urls = set()
    eval_count = 0

    best = {
        "score": -1,
        "reason": "",
        "detected_subject": "",
        "query": "",
        "commons_title": "",
        "file_url": "",
        "local_path": "",
    }

    print("considering ", image_prompt, title)
    # Round 1: initial plan
    current_queries = plan_queries(image_prompt, title=title, topic=topic, n=queries_per_round)
    
    if not current_queries:
        current_queries = [image_prompt][:queries_per_round]
    
    for round_no in range(1, max_rounds + 1):
        print("round", round_no)
        if round_no > 1:
            current_queries = refine_queries(
                image_prompt=image_prompt,
                tried_queries=all_tried_queries,
                best_reason=best["reason"],
                title=title,
                topic=topic,
                n=queries_per_round,
            )
            if not current_queries:
                current_queries = [f"{image_prompt} photograph", f"{image_prompt} illustration"]

        for query in current_queries:
            print("trying query", query)
            all_tried_queries.append(query)
            candidates = commons_search_files(query, limit=candidates_per_query)

            random.shuffle(candidates)
            for cand in candidates:
                url = cand["thumb_url"]
                if url in seen_urls:
                    continue
                seen_urls.add(url)

                stem = f"row{row_index:04d}_eval{eval_count:04d}"
                local_path = download_image(url, DOWNLOAD_DIR, stem)
                if local_path is None:
                    continue

                try:
                    print("scoring", cand['commons_title'])
                    scored = vision_score_candidate(
                        local_path, image_prompt=image_prompt, title=title, topic=topic
                    )
                except Exception as exc:
                    print("image scoring exception", exc)
                    scored = {"score": 0, "reason": f"vision_error: {exc}", "detected_subject": ""}

                eval_count += 1
                if scored["score"] > best["score"]:
                    best = {
                        "score": scored["score"],
                        "reason": scored["reason"],
                        "detected_subject": scored["detected_subject"],
                        "query": query,
                        "commons_title": cand.get("commons_title", ""),
                        "file_url": url,
                        "local_path": str(local_path),
                    }

                if best["score"] >= accept_score:
                    print("accepting image", cand.get("commons_title", ""), accept_score)
                    return {
                        "accepted": True,
                        "rounds_used": round_no,
                        "tries": all_tried_queries,
                        "best": best,
                    }
                print("best so far", best["score"])
                if eval_count >= round_no * max_vision_evals_per_round:
                    break

                if delay_seconds > 0:
                    time.sleep(delay_seconds)

            if eval_count >= round_no * max_vision_evals_per_round:
                break

    return {
        "accepted": best["score"] >= accept_score,
        "rounds_used": max_rounds,
        "tries": all_tried_queries,
        "best": best,
    }


print("Configured models:")
print("TEXT_MODEL  =", TEXT_MODEL)
print("VISION_MODEL=", VISION_MODEL)
print("OLLAMA_API_BASE=", OLLAMA_API_BASE)
print("STANDARD_USER_AGENT=", STANDARD_USER_AGENT)

Task was destroyed but it is pending!
task: <Task pending name='Task-250' coro=<_async_in_context.<locals>.run_in_context() done, defined at /Users/derenrich/.cache/uv/archive-v0/IoiiY86_VILsnkO4HQBl1/lib/python3.13/site-packages/ipykernel/utils.py:57> wait_for=<Task pending name='Task-251' coro=<Kernel.shell_main() running at /Users/derenrich/.cache/uv/archive-v0/IoiiY86_VILsnkO4HQBl1/lib/python3.13/site-packages/ipykernel/kernelbase.py:597> cb=[Task.task_wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at /Users/derenrich/.cache/uv/archive-v0/IoiiY86_VILsnkO4HQBl1/lib/python3.13/site-packages/zmq/eventloop/zmqstream.py:563]>
/Users/derenrich/.local/share/uv/python/cpython-3.13.1-macos-aarch64-none/lib/python3.13/collections/__init__.py:452: RuntimeWarning: coroutine 'Kernel.shell_main' was never awaited
  result = tuple_new(cls, iterable)
Task was destroyed but it is pending!
task: <Task pending name='Task-251' coro=<Kernel.shell_main() running at /Users/derenrich/.cache/

Configured models:
TEXT_MODEL  = ollama/gemma3n
VISION_MODEL= ollama/gemma4:e4b
OLLAMA_API_BASE= http://localhost:11434
STANDARD_USER_AGENT= Moonflower Daniel Erenrich (derenrich@wikimedia.org)


## Run Commons Image Matching
This pipeline consumes the `image` column from `generated_fakes.csv`, generates smarter query variants with a text model, searches Wikimedia Commons, downloads candidates, and uses a vision model to score match quality.

In [27]:
df = pd.read_csv(INPUT_CSV_FILE, dtype=str).fillna("")
if "image" not in df.columns:
    raise ValueError(f"Expected an 'image' column in {INPUT_CSV_FILE}")

work_df = df.head(MAX_ROWS).copy()
print(f"Loaded {len(df)} rows from {INPUT_CSV_FILE}; processing {len(work_df)} rows")

Loaded 150 rows from even_even_easier_generated_fakes_jun15_6632473800.csv; processing 150 rows


In [28]:
results = []

for idx, row in tqdm(work_df.iterrows(), total=len(work_df)):
    image_prompt = row.get("image", "").strip()
    title = row.get("title", "").strip()
    topic = row.get("topic", "").strip()
    first_sentence = row.get("first_sentence", "").strip()

    if not image_prompt:
        results.append({
            "title": title,
            "topic": topic,
            "first_sentence": first_sentence,
            "image_prompt": image_prompt,
            "accepted": False,
            "match_score": -1,
            "match_reason": "missing image prompt",
            "detected_subject": "",
            "commons_query": "",
            "commons_title": "",
            "commons_file_url": "",
            "local_image_path": "",
            "tries_json": "[]",
        })
        continue

    try:
        found = find_best_commons_image(
            image_prompt=image_prompt,
            title=title,
            topic=topic,
            row_index=int(idx),
        )
        best = found["best"]
        results.append({
            "title": title,
            "topic": topic,
            "first_sentence": first_sentence,            
            "image_prompt": image_prompt,
            "accepted": bool(found["accepted"]),
            "rounds_used": int(found["rounds_used"]),
            "match_score": int(best.get("score", -1)),
            "match_reason": best.get("reason", ""),
            "detected_subject": best.get("detected_subject", ""),
            "commons_query": best.get("query", ""),
            "commons_title": best.get("commons_title", ""),
            "commons_file_url": best.get("file_url", ""),
            "local_image_path": best.get("local_path", ""),
            "tries_json": json.dumps(found.get("tries", []), ensure_ascii=False),
        })
    except Exception as exc:
        print(f"pipeline_error: {exc}")
        traceback.print_exception(exc)
        results.append({
            "title": title,
            "topic": topic,
            "first_sentence": first_sentence,            
            "image_prompt": image_prompt,
            "accepted": False,
            "match_score": -1,
            "match_reason": f"pipeline_error: {exc}",
            "detected_subject": "",
            "commons_query": "",
            "commons_title": "",
            "commons_file_url": "",
            "local_image_path": "",
            "tries_json": "[]",
        })
    for k,v in row.items():
        results[-1][k] = v

    

results_df = pd.DataFrame(results)
results_df.head(10)

  0%|          | 0/150 [00:00<?, ?it/s]

considering  A man in a formal suit holding a microphone on a stage Rajesh Vora
round 1
trying query Man microphone stage
scoring File:Man talking on a stage into a microphone in front a building in Moores Creek National Battlefield. Image Number Waso - 943 Co. (279a057756304813bd8fdab1abe8647e).jpg
accepting image File:Man talking on a stage into a microphone in front a building in Moores Creek National Battlefield. Image Number Waso - 943 Co. (279a057756304813bd8fdab1abe8647e).jpg 70
considering  a close-up of a clear plastic disc reflecting light Polycinematic acid
round 1
trying query plastic disc reflection
trying query clear plastic disc
scoring File:Photoelasticity - Clear CD case centre.jpg
accepting image File:Photoelasticity - Clear CD case centre.jpg 70
considering  A scenic view of a rocky canyon with green trees Redstone Gorge State Park
round 1
trying query rocky canyon trees
scoring File:Canyon Trees (11443310454).jpg
accepting image File:Canyon Trees (11443310454).jpg 7

,title,topic,first_sentence,image_prompt,accepted,rounds_used,match_score,match_reason,detected_subject,commons_query,...,image,second_sentence,third_sentence,tell,_seed_title,_seed_topic,_seed_url,_seed_2_title,_seed_2_topic,_seed_2_url
0,Rajesh Vora,Culture.Biography,Rajesh Vora (born 1985) is an Indian theater a...,A man in a formal suit holding a microphone on...,True,1,95,The image shows a man in a formal suit (matchi...,An older man in a suit speaking at a podium wi...,Man microphone stage,...,A man in a formal suit holding a microphone on...,He studied dramatic arts at the Maharaja Sayaj...,Vora's performance in the 2012 musical adaptat...,A single person cannot biologically sing both ...,Pratik Gandhi,Geography.Regions.Asia.South_Asia,https://en.wikipedia.org/wiki/Pratik_Gandhi,Rosemary Clooney,Culture.Media.Music,https://en.wikipedia.org/wiki/Rosemary_Clooney
1,Polycinematic acid,STEM.Chemistry,Polycinematic acid is a heat-resistant synthet...,a close-up of a clear plastic disc reflecting ...,True,1,95,The image is a very detailed close-up of a cle...,"A close-up, highly reflective disc or lens wit...",clear plastic disc,...,a close-up of a clear plastic disc reflecting ...,"Developed by French chemists, the compound for...","Today, it is widely utilized in both the enter...",DVDs and modern DVD players did not exist in 1...,Polytetrafluoroethylene,STEM.STEM*,https://en.wikipedia.org/wiki/Polytetrafluoroe...,Jonah Hill filmography,Culture.Media.Media*,https://en.wikipedia.org/wiki/Jonah_Hill_filmo...
2,Redstone Gorge State Park,Geography.Regions.Americas,Redstone Gorge State Park is a nature reserve ...,A scenic view of a rocky canyon with green trees,True,1,70,The image shows a scenic view of mountains and...,A sweeping panoramic view of rolling hills and...,rocky canyon trees,...,A scenic view of a rocky canyon with green trees,"Established in 1924, the park features sandsto...",It has served as the backdrop for various hist...,"The Academy Awards did not exist in 1850, and ...",Kentucky,Geography.Regions.Americas.North_America,https://en.wikipedia.org/wiki/Kentucky,Sang Heon Lee,Culture.Media.Media*,https://en.wikipedia.org/wiki/Sang_Heon_Lee
3,Charles Sterling,Culture.Media,Charles Sterling (1901–1965) was an English pl...,A vintage black and white portrait of a man in...,True,1,100,The image is indeed a vintage black and white ...,A vintage black and white portrait of a young ...,vintage man portrait,...,A vintage black and white portrait of a man in...,"Known for his sophisticated wit, Sterling wrot...",His productions were filmed live in London and...,"Dailymotion, the internet, and mobile devices ...",Noël Coward,Culture.Media.Media*,https://en.wikipedia.org/wiki/No%C3%ABl_Coward,Dailymotion,Culture.Media.Media*,https://en.wikipedia.org/wiki/Dailymotion
4,Inis Morba,Geography.Regions.Europe,Inis Morba is a small uninhabited island off t...,A rugged green island in the ocean under a clo...,True,1,80,"The image strongly matches 'rugged,' 'green,' ...",A rugged green coastal landscape featuring den...,rugged island photo,...,A rugged green island in the ocean under a clo...,The island covers an area of approximately two...,"Today, it is protected as a national nature re...",Television and reality TV shows did not exist ...,Republic of Ireland,Geography.Regions.Europe.Europe*,https://en.wikipedia.org/wiki/Republic_of_Ireland,Erica Mena,Culture.Media.Media*,https://en.wikipedia.org/wiki/Erica_Mena
5,Clara Montgomery,Culture.Biography,Clara Montgomery (born 1980) is an American ac...,A black and white photograph of a woman lookin...,True,1,95,The image is a black and white photograph feat...,A photograph of a woman seated in an interior ...,woman portrait black white,...,A black and white photograph of a woman lookin...,"She later transitioned to Broadway, receiving ...","Throughout her career, she won multiple awards...",A person born in 1980 cannot star in a movie r...,Marcia Gay Harden,Culture.Biography.Women,htt

In [29]:
Path(OUTPUT_JSON_FILE).write_text(
    json.dumps(results, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
results_df.to_csv(OUTPUT_CSV_FILE, index=False)

print(f"Wrote {len(results_df)} rows to {OUTPUT_CSV_FILE}")
print(f"Wrote detailed JSON to {OUTPUT_JSON_FILE}")

summary = {
    "accepted": int(results_df["accepted"].fillna(False).sum()),
    "total": int(len(results_df)),
    "avg_score": float(pd.to_numeric(results_df["match_score"], errors="coerce").fillna(0).mean()),
}

Wrote 150 rows to even_even_easier_generated_fakes_jun15_with_commons_matches_6632473800.csv
Wrote detailed JSON to even_even_easier_generated_fakes_jun15_with_commons_matches_6632473800.json


In [30]:
URL = "https://en.wikipedia.org/w/rest.php/attribution/v0-beta/pages/File:{}/signals"
HEADERS = {"User-Agent": "derenrich@wikimedia.org - gacha attribution data"}

MAX_WORKERS = 4
MAX_RETRIES = 5
INITIAL_BACKOFF = 1.0

def fetch_attribution(image_url):
    if not image_url:
        return {}
    if '/thumb/' in image_url:
        image_filename = image_url.split("/")[-2]
    else:
        image_filename = image_url.split("/")[-1]
    if not image_filename:
        return {}
    
    backoff = INITIAL_BACKOFF
    for attempt in range(MAX_RETRIES):
        try:
            resp = requests.get(URL.format(image_filename), headers=HEADERS, timeout=10)
            if resp.status_code in (429, 500, 502, 503, 504):
                retry_after = resp.headers.get("Retry-After")
                sleep_for = float(retry_after) if retry_after else backoff + random.uniform(0, backoff * 0.1)
                time.sleep(sleep_for)
                backoff = min(backoff * 2, 60)
                continue
            resp.raise_for_status()
            data = resp.json()
            break
        except (requests.ConnectionError, requests.Timeout):
            print("timeout/connection")
            time.sleep(backoff + random.uniform(0, backoff * 0.5))
            backoff = min(backoff * 2, 60)
        except Exception as e:
            print("Exception", e, image_url, image_filename)
            return {}
    else:
        print("exhausted retries")
        return {}

    link = data.get("essential", {}).get("link")
    license = data.get("essential", {}).get("license", {}).get("title")
    author = data.get("essential", {}).get("credit")
    return dict(link=link, license=license, author=author)


with open(OUTPUT_CSV_FILE, newline="") as f:
     rows = list(csv.DictReader(f))


url_to_attribution: dict[str, dict] = {}
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(fetch_attribution, row['commons_file_url']): row['commons_file_url'] for row in rows}
    for fut in tqdm(as_completed(futures), total=len(futures), desc="Fetching attribution"):
        url = futures[fut]
        url_to_attribution[url] = fut.result()


fieldnames = list(rows[0].keys()) + ["image_license", "image_credit", "image_attribution_url"]
with open(OUTPUT_CSV_FILE_WITH_ATRIB, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    for r in rows:
        attribution_info = url_to_attribution.get(r["commons_file_url"], {})
        r['image_license'] = attribution_info.get('license')
        r['image_credit'] = attribution_info.get('author')
        r['image_attribution_url'] = attribution_info.get('link')                
        writer.writerow(r)

print("wrote ", OUTPUT_CSV_FILE_WITH_ATRIB)

Fetching attribution:   0%|          | 0/150 [00:00<?, ?it/s]

wrote  even_even_easier_generated_fakes_jun15_with_commons_matches_and_atrib_6632473800.csv
